# 19 — Pull Cline Center Coup d'État Project Dataset

**Source:** Cline Center Coup d'État Project Dataset v2.2.2  
**Provider:** Cline Center for Advanced Social Research, University of Illinois Urbana-Champaign  
**Access:** Public — Illinois Data Bank (IDB-3143201)  
**Coverage:** Global, 1945–early 2026  
**Frequency:** Event-level (each row is a single coup event)  
**Total events:** 1,161 (472 realized, 408 attempted, 281 conspiracies)  

## What this notebook pulls

The Cline Center Coup d'État Project is the world's largest global registry of failed and
successful coups, covering every country since 1945. It extends and supersedes the
Powell-Thyne dataset (notebook 04) by:

- Adding a **conspiracy** category for planned-but-thwarted events absent from Powell-Thyne
- Documenting the **fate of the deposed executive** (killed, jailed, fled, etc.)
- Flagging **popular revolts** and **counter-coups** as subtypes
- Providing a `coup_id` that links each event to 365 pages of source documentation
- Updating through early 2026 (Powell-Thyne lags by ~2 years)

This feeds **Domain 2 — Governance/Institutions** predictors in the meta-plan
(coup onset labels, irregular leadership change, executive fate).

### Key variables (29 total in v2.2.2)

| Variable | Description |
|---|---|
| `country` | Country name |
| `ccode` | Correlates of War (CoW) country code |
| `year` | Year of coup event |
| `month` | Month of coup event |
| `day` | Day of coup event |
| `coup_id` | Unique event ID linking to source documentation |
| `event_type` | 1 = realized coup; 2 = attempted coup; 3 = conspiracy |
| `realized` | 1 if event_type == 1 (realized/successful coup) |
| `attempted` | 1 if event_type == 2 (failed attempt) |
| `conspiracy` | 1 if event_type == 3 (planned, thwarted before action) |
| `popular` | Popular revolt involved (0/1) |
| `counter` | Counter-coup (0/1) |
| `noharm` | Deposed executive not harmed (0/1) |
| `injured` | Deposed executive injured (0/1) |
| `killed` | Deposed executive killed (0/1) |
| `harrest` | Deposed executive under house arrest (0/1) |
| `jailed` | Deposed executive arrested/detained/jailed (0/1) |
| `tried` | Deposed executive tried in court (0/1) |
| `fled` | Deposed executive fled the country (0/1) |

### Derived country-year panel variables
- `coup_any` — any coup event of any type this country-year
- `coup_realized` — any successful coup this country-year
- `coup_attempt` — any attempt (realized or attempted, excluding conspiracies)
- `coup_conspiracy` — any documented conspiracy this country-year
- `coup_event_count` — total number of coup events (all types)
- `executive_killed` — executive killed in any coup event this country-year
- `popular_coup` — popular revolt involved in any coup event this country-year
- `counter_coup` — counter-coup involved in any coup event this country-year

## Access note

The dataset is freely available on the Illinois Data Bank. If the automated download
fails, download `Coup_Data_v2.2.2.csv` (or the current version zip) manually from:

  https://databank.illinois.edu/datasets/IDB-3143201

and place the CSV at `coup_files/Coup_Data_v2.2.2.csv`, then re-run from **Load data**.

```
coup_files/Coup_Data_v2.2.2.csv     ← manual fallback
```

## Citation

Peyton, Buddy; Bajjalieh, Joseph; Martin, Michael; Alahi, Sam; Fadell, Norah; and
Jeralds, Maddie. 2026. "Cline Center Coup d'État Project Dataset". V.2.2.2.
February 17. University of Illinois Urbana-Champaign. https://doi.org/10.13012/B2IDB-3143201_V10

## Required environment variables
```
ADLS_ACCOUNT_NAME  — Azure storage account name
ADLS_CONTAINER     — Container name (default: 'data')
```

In [ ]:
import os
import io
import zipfile
import requests
import pandas as pd
from pathlib import Path
from datetime import datetime
from azure.identity import DefaultAzureCredential

## Configuration

In [ ]:
ADLS_ACCOUNT_NAME = os.environ["ADLS_ACCOUNT_NAME"]
ADLS_CONTAINER    = os.getenv("ADLS_CONTAINER", "data")
RUN_DATE          = datetime.utcnow().strftime("%Y%m%d")

# Cline Center Coup d'État Project v2.2.2
# Illinois Data Bank: https://databank.illinois.edu/datasets/IDB-3143201
# The download_bundle endpoint returns a ZIP of all dataset files
CLINE_COUP_URL = (
    "https://databank.illinois.edu/datasets/IDB-3143201/download_bundle"
)

# Expected CSV filename inside the bundle (update if Illinois Data Bank renames it)
CLINE_COUP_CSV_PATTERN = "coup"  # case-insensitive substring match

# Local fallback: place the CSV here if automated download fails
CLINE_LOCAL_DIR  = Path("coup_files")
CLINE_LOCAL_FILE = CLINE_LOCAL_DIR / "Coup_Data_v2.2.2.csv"

print(f"Run date      : {RUN_DATE}")
print(f"Primary URL   : {CLINE_COUP_URL}")
print(f"Local fallback: {CLINE_LOCAL_FILE} (exists={CLINE_LOCAL_FILE.exists()})")

## ADLS helper

In [ ]:
credential = DefaultAzureCredential()
storage_options = {
    "account_name": ADLS_ACCOUNT_NAME,
    "credential": credential,
}

def adls_path(subpath: str) -> str:
    return (
        f"abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}"
        f".dfs.core.windows.net/{subpath}"
    )

def write_parquet(df: pd.DataFrame, subpath: str) -> None:
    path = adls_path(subpath)
    df.to_parquet(path, storage_options=storage_options, index=False, engine="pyarrow")
    print(f"  Written {len(df):,} rows → {path}")

## Load Cline Center coup data

Tries to download the bundle ZIP from the Illinois Data Bank, extracts the CSV, and reads
it into a DataFrame. Falls back to a local file if the download is unavailable.

In [ ]:
def load_cline_from_url(url: str, csv_pattern: str) -> pd.DataFrame | None:
    print(f"Downloading Cline Center coup data from {url} ...")
    try:
        resp = requests.get(url, timeout=120)
        resp.raise_for_status()
    except Exception as e:
        print(f"  Download failed: {e}")
        return None

    content_type = resp.headers.get("content-type", "")
    content = resp.content

    # Illinois Data Bank bundle → ZIP archive containing the CSV
    if "zip" in content_type or content[:4] == b"PK\x03\x04":
        try:
            with zipfile.ZipFile(io.BytesIO(content)) as z:
                names = z.namelist()
                csv_name = next(
                    (n for n in names if csv_pattern.lower() in n.lower() and n.lower().endswith(".csv")),
                    None,
                )
                if csv_name is None:
                    # Fall back to any CSV in the archive
                    csv_name = next((n for n in names if n.lower().endswith(".csv")), None)
                if csv_name is None:
                    print(f"  No CSV found in ZIP. Contents: {names}")
                    return None
                print(f"  Extracting: {csv_name}")
                with z.open(csv_name) as f:
                    return pd.read_csv(f, low_memory=False)
        except Exception as e:
            print(f"  Failed to parse ZIP: {e}")
            return None

    # Direct CSV response
    try:
        return pd.read_csv(io.BytesIO(content), low_memory=False)
    except Exception as e:
        print(f"  Failed to parse response as CSV: {e}")
        return None


def load_cline_from_local(path: Path) -> pd.DataFrame | None:
    if not path.exists():
        # Also try any CSV in the directory
        candidates = list(path.parent.glob("*.csv")) if path.parent.exists() else []
        if candidates:
            path = candidates[0]
            print(f"  Using local file: {path}")
        else:
            return None
    print(f"Loading from local file: {path}")
    try:
        return pd.read_csv(str(path), low_memory=False)
    except Exception as e:
        print(f"  Failed to read local CSV: {e}")
        return None


df_coup_raw = load_cline_from_url(CLINE_COUP_URL, CLINE_COUP_CSV_PATTERN)

if df_coup_raw is None:
    df_coup_raw = load_cline_from_local(CLINE_LOCAL_FILE)

if df_coup_raw is None:
    print(
        "\nWARNING: Cline Center coup data not available.\n"
        "  Download the dataset manually from:\n"
        "  https://databank.illinois.edu/datasets/IDB-3143201\n"
        f"  and place the CSV at: {CLINE_LOCAL_FILE}\n"
    )
else:
    df_coup_raw.columns = [c.lower().strip() for c in df_coup_raw.columns]
    print(f"Raw shape : {df_coup_raw.shape}")
    print(f"Columns   : {list(df_coup_raw.columns)}")
    df_coup_raw.head(3)

## Schema and type casting

The Cline Center CSV uses numeric codes for `event_type`:
- `1` — realized (successful coup; authority of target seized or removed)
- `2` — attempted (action taken but failed)
- `3` — conspiracy (planned but thwarted before action taken)

We preserve the raw numeric code and derive explicit boolean columns for each type.

In [ ]:
# Documented schema fields from Cline Center Codebook v2.2.2
INT_COLS = [
    "ccode",       # CoW country code
    "year",        # Year of event
    "month",       # Month of event
    "day",         # Day of event
    "event_type",  # 1=realized, 2=attempted, 3=conspiracy
    "realized",    # 1 if successful coup
    "attempted",   # 1 if failed attempt
    "conspiracy",  # 1 if thwarted before action
    "popular",     # Popular revolt involved
    "counter",     # Counter-coup
    "noharm",      # Deposed executive not harmed
    "injured",     # Deposed executive injured
    "killed",      # Deposed executive killed
    "harrest",     # Deposed executive under house arrest
    "jailed",      # Deposed executive jailed
    "tried",       # Deposed executive tried in court
    "fled",        # Deposed executive fled
]

if df_coup_raw is not None:
    df_coup = df_coup_raw.copy()

    for col in INT_COLS:
        if col in df_coup.columns:
            df_coup[col] = pd.to_numeric(df_coup[col], errors="coerce").astype("Int64")

    # If event_type is present but realized/attempted/conspiracy dummies are not,
    # derive them from event_type
    if "event_type" in df_coup.columns:
        if "realized" not in df_coup.columns:
            df_coup["realized"]   = (df_coup["event_type"] == 1).astype("Int8")
        if "attempted" not in df_coup.columns:
            df_coup["attempted"]  = (df_coup["event_type"] == 2).astype("Int8")
        if "conspiracy" not in df_coup.columns:
            df_coup["conspiracy"] = (df_coup["event_type"] == 3).astype("Int8")

    # Construct event_date where date components are available
    date_cols = [c for c in ["year", "month", "day"] if c in df_coup.columns]
    if len(date_cols) == 3:
        df_coup["event_date"] = pd.to_datetime(
            df_coup[["year", "month", "day"]].rename(
                columns={"year": "year", "month": "month", "day": "day"}
            ),
            errors="coerce",
        )

    # Validate expected schema
    missing = [c for c in INT_COLS if c not in df_coup.columns]
    if missing:
        print(f"Note: {len(missing)} expected column(s) not in this version: {missing}")

    print(f"Coup events : {len(df_coup):,}")
    if "year" in df_coup.columns:
        print(f"Year range  : {df_coup['year'].min()}–{df_coup['year'].max()}")
    if "realized" in df_coup.columns:
        n_realized   = int(df_coup["realized"].sum())
        n_attempted  = int(df_coup["attempted"].sum()) if "attempted" in df_coup.columns else "n/a"
        n_conspiracy = int(df_coup["conspiracy"].sum()) if "conspiracy" in df_coup.columns else "n/a"
        print(f"  Realized   : {n_realized:,}")
        print(f"  Attempted  : {n_attempted}")
        print(f"  Conspiracy : {n_conspiracy}")
    df_coup.head(3)
else:
    df_coup = pd.DataFrame()

## Write event-level data to ADLS

In [ ]:
if not df_coup.empty:
    write_parquet(df_coup, f"raw/cline_coup/{RUN_DATE}/cline_coup_events.parquet")

## Derive country-year panel

Aggregate event-level coup records to a **country-year** panel for joining into the
main feature matrix (`02/02_build_feature_matrix`). Each row represents one country-year; the binary
flags indicate whether any coup event of each type occurred in that country-year.

The `coup_attempt` column (realized + attempted, excluding conspiracies) is the closest
match to the Powell-Thyne definition used in notebook 04, enabling direct comparison.

In [ ]:
if not df_coup.empty and "year" in df_coup.columns:
    # Identify the country identifier column (CoW code or country name)
    country_keys = [c for c in ["ccode", "country"] if c in df_coup.columns]
    if not country_keys:
        raise ValueError("No country identifier column found (expected 'ccode' or 'country')")

    group_keys = country_keys + ["year"]

    def _any(s):
        return int((pd.to_numeric(s, errors="coerce").fillna(0) > 0).any())

    agg_spec = {
        "coup_event_count": ("year", "count"),
    }

    for col, out_col in [
        ("realized",   "coup_realized"),
        ("attempted",  "coup_attempted"),
        ("conspiracy", "coup_conspiracy"),
        ("popular",    "popular_coup"),
        ("counter",    "counter_coup"),
        ("killed",     "executive_killed"),
        ("fled",       "executive_fled"),
        ("jailed",     "executive_jailed"),
    ]:
        if col in df_coup.columns:
            agg_spec[out_col] = (col, _any)

    df_country_year = (
        df_coup
        .groupby(group_keys, as_index=False)
        .agg(**agg_spec)
    )

    # coup_attempt: realized or attempted (matches Powell-Thyne scope)
    if "coup_realized" in df_country_year.columns and "coup_attempted" in df_country_year.columns:
        df_country_year["coup_attempt"] = (
            (df_country_year["coup_realized"].fillna(0).astype(int) |
             df_country_year["coup_attempted"].fillna(0).astype(int))
        ).clip(upper=1).astype("Int64")

    # coup_any: any event type
    type_cols = [c for c in ["coup_realized", "coup_attempted", "coup_conspiracy"] if c in df_country_year.columns]
    if type_cols:
        df_country_year["coup_any"] = (
            df_country_year[type_cols].fillna(0).astype(int).max(axis=1)
        ).clip(upper=1).astype("Int64")

    for col in ["coup_event_count"] + [c for c in df_country_year.columns if c not in group_keys + ["coup_event_count"]]:
        if col in df_country_year.columns:
            df_country_year[col] = pd.to_numeric(df_country_year[col], errors="coerce").astype("Int64")

    print(f"Country-year panel : {len(df_country_year):,} rows")
    print(f"Year range         : {df_country_year['year'].min()}–{df_country_year['year'].max()}")
    if "ccode" in df_country_year.columns:
        print(f"Countries (ccode)  : {df_country_year['ccode'].nunique()}")
    if "coup_any" in df_country_year.columns:
        print(f"Country-years with any coup event: {int(df_country_year['coup_any'].sum())}")
    if "coup_realized" in df_country_year.columns:
        print(f"  Realized coups   : {int(df_country_year['coup_realized'].sum())}")
    if "coup_attempted" in df_country_year.columns:
        print(f"  Attempted coups  : {int(df_country_year['coup_attempted'].sum())}")
    if "coup_conspiracy" in df_country_year.columns:
        print(f"  Conspiracies     : {int(df_country_year['coup_conspiracy'].sum())}")
    df_country_year.head(3)
else:
    df_country_year = pd.DataFrame()

In [ ]:
if not df_country_year.empty:
    write_parquet(df_country_year, f"raw/cline_coup/{RUN_DATE}/cline_coup_country_year.parquet")

## Summary

In [ ]:
print("=" * 55)
print("Cline Center Coup d'État Project pull complete")
print("=" * 55)
if not df_coup.empty:
    print(f"  Coup events        : {len(df_coup):,}")
    if "year" in df_coup.columns:
        print(f"  Year range         : {df_coup['year'].min()}–{df_coup['year'].max()}")
if not df_country_year.empty:
    print(f"  Country-year rows  : {len(df_country_year):,}")
    if "coup_any" in df_country_year.columns:
        print(f"  Country-years with coup event: {int(df_country_year['coup_any'].sum())}")
print()
print("ADLS paths written:")
print(f"  raw/cline_coup/{RUN_DATE}/cline_coup_events.parquet")
print(f"  raw/cline_coup/{RUN_DATE}/cline_coup_country_year.parquet")